# CNN for Image Classification

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision # library for CV
from torchvision.datasets import CIFAR10 # importing Dataset

## Datasets & DataLoaders

In [2]:
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

trainset = CIFAR10(root="./data", train=True, download=True, transform=transform)
testset = CIFAR10(root="./data", train=False, download=True, transform=transform)

In [3]:
trainloader = DataLoader(trainset, batch_size=64, shuffle=True)
testloader = DataLoader(testset, batch_size=64)

## Build the CNN

In [4]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # kernel size = 2, stride = 2

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2) ,

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(4*4*128, 256),
            nn.ReLU(),

            nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1) # flattening
        x = self.fc_layers(x)

        return x

In [5]:
model = CNN()

In [6]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

## Training the CNN

In [7]:
epochs = 10

for epoch in range(epochs):
    epoch_training_loss = 0.0

    for images, labels in trainloader:
        optimizer.zero_grad()
        
        output = model.forward(images) # FP
        loss = criterion(output, labels) # loss fnx
        loss.backward() # BP
        optimizer.step() # update params

        epoch_training_loss += loss.item()

    print(f"epoch={epoch+1}/{epochs} & loss={epoch_training_loss/len(trainloader)}")

epoch=1/10 & loss=1.3813411065989443
epoch=2/10 & loss=0.9649330985058299
epoch=3/10 & loss=0.7784149339589317
epoch=4/10 & loss=0.648862781465206
epoch=5/10 & loss=0.5430738731570866
epoch=6/10 & loss=0.44636179546794624
epoch=7/10 & loss=0.3621375569335335
epoch=8/10 & loss=0.2855228259016181
epoch=9/10 & loss=0.22246825564509767
epoch=10/10 & loss=0.17861388032527073


## Evaludate our CNN

In [8]:
correct_labels = 0
total_labels = 0

model.eval()

with torch.no_grad():
    for images, labels in testloader:
        outputs = model.forward(images)
        _, predicted  = torch.max(outputs, 1)

        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)

print(f"accuracy = {correct_labels / total_labels * 100}")

accuracy = 75.27000000000001


## User input and image prediction

In [9]:
import gradio as gr
from PIL import Image

classes = ['airplane', 'automobile', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck']

predict_transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transform
])

def predict(img):
    img = img.convert("RGB")
    x = predict_transform(img).unsqueeze(0)
    model.eval()
    with torch.no_grad():
        probs = torch.softmax(model(x), dim=1)[0]
    return {classes[i]: float(probs[i]) for i in range(10)}

def predict_path(path):
    return predict(Image.open(path.strip().strip('"').strip("'")))

with gr.Blocks() as demo:
    gr.Markdown("## CIFAR-10 Image Classifier")
    with gr.Tab("Drag & drop"):
        img_in = gr.Image(type="pil", label="Drop an image here")
        out1 = gr.Label(num_top_classes=3)
        img_in.change(predict, img_in, out1)
    with gr.Tab("Image path"):
        path_in = gr.Textbox(label="Image path", placeholder="C:/Users/me/Pictures/cat.jpg")
        out2 = gr.Label(num_top_classes=3)
        path_in.submit(predict_path, path_in, out2)

demo.launch(inline=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
